# 📈 QuantVision — Train Global LSTM Models

**Instructions:**
1. Go to `Runtime → Change runtime type → Hardware accelerator → GPU`
2. Run all cells top-to-bottom
3. The last cell will download 4 files — move them to your local `models/` folder

In [ ]:
# Cell 1 — Install only what we need (NO sklearn)
!pip install -q yfinance pandas-ta

In [ ]:
# Cell 2 — Imports (pure numpy scaler — no sklearn)
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib, os

# Pure numpy MinMaxScaler — avoids ALL sklearn/numpy version conflicts
class TorchScaler:
    def fit(self, X):
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        return self
    def transform(self, X):
        return (X - self.min_) / (self.max_ - self.min_ + 1e-8)
    def fit_transform(self, X):
        return self.fit(X).transform(X)
    def inverse_transform(self, X):
        return X * (self.max_ - self.min_ + 1e-8) + self.min_

MinMaxScaler = TorchScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {device}')
os.makedirs('models', exist_ok=True)

## Step 1 — Download 10 years of data across 10 major assets

In [ ]:
# Cell 3 — Download stock data
TICKERS = ['AAPL', 'MSFT', 'NVDA', 'TSLA', 'RELIANCE.NS',
           'HDFCBANK.NS', 'TCS.NS', '^NSEI', 'BTC-USD', 'ETH-USD']

datasets = []
for ticker in TICKERS:
    print(f'  ⬇️  {ticker}...', end=' ')
    try:
        df = yf.download(ticker, period='10y', progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel(1)
        if len(df) < 200:
            print('SKIP (too short)')
            continue
        df['Log_Return']   = np.log(df['Close'] / df['Close'].shift(1))
        df['Realized_Vol'] = df['Log_Return'].rolling(21).std() * np.sqrt(252) * 100
        df = df.dropna()
        datasets.append(df)
        print(f'{len(df)} rows ✓')
    except Exception as e:
        print(f'ERROR: {e}')

print(f'\n✅ {len(datasets)} assets loaded, total rows: {sum(len(d) for d in datasets)}')

## Step 2 — Train Global Neural Volatility LSTM

In [ ]:
# Cell 4 — Train volatility LSTM
class NeuralVol(nn.Module):
    def __init__(self, input_size=1, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# Global scaler fitted on ALL assets' volatility combined
all_rv = np.concatenate([d['Realized_Vol'].values for d in datasets]).reshape(-1, 1)
vol_scaler = TorchScaler()
vol_scaler.fit(all_rv)

LOOKBACK = 20
X_vol, y_vol = [], []
for df in datasets:
    rv = vol_scaler.transform(df['Realized_Vol'].values.reshape(-1, 1)).flatten()
    for i in range(LOOKBACK, len(rv)):
        X_vol.append(rv[i-LOOKBACK:i])
        y_vol.append(rv[i])

X_v = torch.FloatTensor(np.array(X_vol).reshape(-1, LOOKBACK, 1)).to(device)
y_v = torch.FloatTensor(np.array(y_vol).reshape(-1, 1)).to(device)

vol_model = NeuralVol().to(device)
opt_v = torch.optim.Adam(vol_model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

print(f'Training on {len(X_vol):,} samples...')
for ep in range(150):
    opt_v.zero_grad()
    loss = loss_fn(vol_model(X_v), y_v)
    loss.backward()
    opt_v.step()
    if (ep+1) % 30 == 0:
        print(f'  Epoch {ep+1}/150  Loss={loss.item():.5f}')

torch.save(vol_model.state_dict(), 'models/global_volatility_lstm.pt')
joblib.dump(vol_scaler, 'models/global_vol_scaler.pkl')
print('✅ Volatility model saved!')

## Step 3 — Train 7-Day & 30-Day Price Forecasters

In [ ]:
# Cell 5 — Train price forecasters
class LSTMForecaster(nn.Module):
    def __init__(self, input_size=2, horizon=7, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, num_layers=2, batch_first=True, dropout=0.2)
        self.head_lo = nn.Linear(hidden, horizon)
        self.head_md = nn.Linear(hidden, horizon)
        self.head_hi = nn.Linear(hidden, horizon)
    def forward(self, x):
        ctx = self.lstm(x)[0][:, -1, :]
        return self.head_lo(ctx), self.head_md(ctx), self.head_hi(ctx)

def q_loss(pred, tgt, q):
    e = tgt - pred
    return torch.max((q-1)*e, q*e).mean()

def train_forecaster(horizon, epochs=100):
    print(f'\n⏳ Training {horizon}-day forecaster...')
    PRICE_LB = 30
    X_all, y_all = [], []
    for df in datasets:
        data = df[['Close', 'Log_Return']].values
        sc = TorchScaler()
        ds = sc.fit_transform(data)
        for i in range(PRICE_LB, len(ds) - horizon + 1):
            X_all.append(ds[i-PRICE_LB:i])
            y_all.append(ds[i:i+horizon, 0])

    X_t = torch.FloatTensor(np.array(X_all)).to(device)
    y_t = torch.FloatTensor(np.array(y_all)).to(device)
    model = LSTMForecaster(input_size=2, horizon=horizon).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=5e-4)

    print(f'  Samples: {len(X_all):,}')
    for ep in range(epochs):
        opt.zero_grad()
        lo, md, hi = model(X_t)
        loss = q_loss(lo, y_t, 0.1) + q_loss(md, y_t, 0.5) + q_loss(hi, y_t, 0.9)
        loss.backward()
        opt.step()
        if (ep+1) % 25 == 0:
            print(f'  Epoch {ep+1}/{epochs}  Loss={loss.item():.5f}')

    path = f'models/global_forecast_lstm_{horizon}d.pt'
    torch.save(model.state_dict(), path)
    print(f'  ✅ Saved {path}')

train_forecaster(7)
train_forecaster(30)
print('\n🎉 All models trained and saved!')

## Step 4 — Download the 4 model files
Place all downloaded files into the `models/` folder of your local project.

In [ ]:
# Cell 6 — Download trained model files
try:
    from google.colab import files
    import time
    for f in ['models/global_volatility_lstm.pt',
              'models/global_vol_scaler.pkl',
              'models/global_forecast_lstm_7d.pt',
              'models/global_forecast_lstm_30d.pt']:
        files.download(f)
        time.sleep(2)
    print('✅ All 4 files downloaded!')
except ImportError:
    print('Files saved in the models/ folder.')